# NUTDTS 816 Time Series Analysis
## L23 Industry and healthcare applications

Lab notebook for Chapter 12 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 12.1 Demand and energy forecasting

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import MSTL
import tsdata
dem = tsdata.elecdemand()        # half-hourly electricity demand (GW), Victoria, 2014, with temperature and work-day flag (fpp2)
y = dem['Demand']['2014-06-01':'2014-07-31']      # two winter months: daily and weekly cycles are clear
res = MSTL(y, periods=(48, 336), stl_kwargs={'seasonal_deg': 0}).fit()
fig = res.plot(); fig.set_size_inches(10, 8)
_caption = 'MSTL on half-hourly demand: the daily cycle (period 48), the weekly cycle (period 336, weekend troughs), a trend, and a remainder that still contains weather-driven variation.'

In [ ]:
# Temperature as a driver: demand against temperature, by work day
fig, ax = plt.subplots(figsize=(7, 3.2))
d = dem.resample('D').agg({'Demand': 'sum', 'Temperature': 'mean', 'WorkDay': 'first'})
for wd, lab, col in [(1, 'work day', '#1B5E3A'), (0, 'weekend / holiday', '#B8860B')]:
    sub = d[d.WorkDay == wd]; ax.scatter(sub.Temperature, sub.Demand, s=8, color=col, label=lab)
ax.set_xlabel('daily mean temperature (°C)'); ax.set_ylabel('daily demand (GW·half-hours)'); ax.set_title('Daily demand against temperature, Victoria 2014'); ax.legend(fontsize=8)
_caption = 'The U-shape: demand rises in cold weather (heating) and in hot weather (cooling), with a minimum around 18 °C, and work days sit above weekends at every temperature. A load model needs temperature as a nonlinear regressor and a work-day indicator.'

### 12.2 Intermittent demand and Croston's method

In [ ]:
rng = np.random.default_rng(12); n = 120
demand = np.where(rng.random(n) < 0.2, rng.poisson(6, n), 0)      # 20% of weeks have a sale; sizes around 6
def croston(x, alpha=0.1):
    z, p = x[x > 0][0], np.argmax(x > 0) + 1; q = 1; fc = []
    for t in range(len(x)):
        fc.append(z / p)
        if x[t] > 0: z = alpha * x[t] + (1 - alpha) * z; p = alpha * q + (1 - alpha) * p; q = 1
        else: q += 1
    return np.array(fc)
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
ses = SimpleExpSmoothing(pd.Series(demand.astype(float)), initialization_method='estimated').fit(smoothing_level=0.1, optimized=False).fittedvalues
fig, ax = plt.subplots(figsize=(9, 3)); ax.bar(range(n), demand, color='#cccccc', label='weekly demand'); ax.plot(croston(demand), color='#1B5E3A', lw=1.8, label='Croston demand rate'); ax.plot(ses.values, color='#A0302A', lw=1.2, label='SES on the raw series')
ax.legend(fontsize=8); ax.set_xlabel('week'); ax.set_title('Intermittent demand: Croston forecasts a stable rate; SES saws up and down')
print(f'True demand rate = {0.2 * 6:.2f} per week; Croston final rate = {croston(demand)[-1]:.2f}; SES final level = {ses.iloc[-1]:.2f}')
_caption = 'Croston forecasts the rate of demand; smoothing the raw series follows the last few zeros and sales and is wrong for stocking.'

### 12.3 Production decline-curve analysis

In [ ]:
from scipy.optimize import curve_fit
t = np.arange(1, 61); qi, Di, b = 1200.0, 0.08, 0.6
rng = np.random.default_rng(45); e = np.zeros(60)
for k in range(1, 60): e[k] = 0.5 * e[k-1] + rng.normal(0, 0.06)
q = qi * (1 + b * Di * t) ** (-1 / b) * np.exp(e)                          # synthetic hyperbolic decline with AR(1) noise (bbl/day)
hyp = lambda t, qi, Di, b: qi * (1 + b * Di * t) ** (-1 / b); exp_ = lambda t, qi, Di: qi * np.exp(-Di * t)
fit_len = 24
ph, _ = curve_fit(hyp, t[:fit_len], q[:fit_len], p0=[1000, 0.05, 0.5], bounds=([0, 0, 0.01], [1e5, 2, 1.5]))
pe, _ = curve_fit(exp_, t[:fit_len], q[:fit_len], p0=[1000, 0.05])
fig, ax = plt.subplots(figsize=(9, 3.2)); ax.plot(t, q, 'o', ms=3, color='#555555', label='monthly rate (synthetic well)')
tt = np.arange(1, 121); ax.plot(tt, hyp(tt, *ph), color='#1B5E3A', lw=1.8, label=f'hyperbolic fit on 24 months (b = {ph[2]:.2f})'); ax.plot(tt, exp_(tt, *pe), color='#A0302A', lw=1.4, ls='--', label='exponential fit on 24 months')
ax.plot(tt, hyp(tt, qi, Di, b), color='#B8860B', lw=1, ls=':', label='true curve'); ax.axvline(fit_len, color='#555555', lw=0.8); ax.set_yscale('log'); ax.set_xlabel('month'); ax.set_ylabel('bbl/day (log scale)'); ax.legend(fontsize=8); ax.set_title('Decline-curve analysis as a trend-extrapolation problem')
eur = lambda f, p: np.sum(f(np.arange(1, 361), *p)) * 30.4
print(f'Ten-year cumulative production: true {eur(hyp, (qi, Di, b))/1e3:,.0f} kbbl | hyperbolic fit {eur(hyp, ph)/1e3:,.0f} kbbl | exponential fit {eur(exp_, pe)/1e3:,.0f} kbbl')
_caption = 'Two models that fit the first two years almost identically diverge at ten years: the hyperbolic fit underestimates cumulative production by about a quarter and the exponential by nearly 40%. The reserves estimate depends on the functional form and on b, which two years of data cannot pin down: the same lesson as every long-horizon forecast in this course.'

### 12.4 Count series in public health

In [ ]:
import statsmodels.api as sm
cases = tsdata.nigeria_malaria(); rain = tsdata.nigeria_rainfall()        # simulated weekly cases and monthly rainfall
rain_w = rain.resample('W-SUN').interpolate().reindex(cases.index).interpolate()   # rainfall on the weekly grid
t = np.arange(len(cases)); X = pd.DataFrame(index=cases.index)
for k in (1, 2): X[f's{k}'] = np.sin(2 * np.pi * k * t / 52.18); X[f'c{k}'] = np.cos(2 * np.pi * k * t / 52.18)
X['trend'] = t / 52; X['lag1'] = np.log1p(cases.shift(1)); X['lag2'] = np.log1p(cases.shift(2)); X['rain_l8'] = np.log1p(rain_w.shift(8))
data = pd.concat([cases, X], axis=1).dropna(); Xd = sm.add_constant(data.drop(columns='cases'))
train = data.index < '2026-01-01'
pois = sm.GLM(data['cases'][train], Xd[train], family=sm.families.Poisson()).fit()
alpha_nb = max(((data['cases'][train] - pois.fittedvalues)**2 - pois.fittedvalues).sum() / (pois.fittedvalues**2).sum(), 0.01)   # moment estimate of dispersion
nb = sm.GLM(data['cases'][train], Xd[train], family=sm.families.NegativeBinomial(alpha=alpha_nb)).fit()
print(f'Poisson deviance / df = {pois.deviance / pois.df_resid:.2f} (1 if no over-dispersion)   ->  negative binomial with dispersion alpha = {alpha_nb:.3f}')
print(nb.summary().tables[1])

In [ ]:
# One-step-ahead forecasts for 2026 with 80% intervals from the fitted negative binomial, and a Gaussian comparison
from scipy import stats
mu_nb = nb.predict(Xd[~train]); size = 1 / alpha_nb
lo, hi = stats.nbinom.ppf(0.1, size, size / (size + mu_nb)), stats.nbinom.ppf(0.9, size, size / (size + mu_nb))
fig, ax = plt.subplots(figsize=(9, 3.2)); cases['2024-07':].plot(ax=ax, lw=1, label='weekly cases (simulated)')
pd.Series(mu_nb.values, index=mu_nb.index).plot(ax=ax, lw=2, color='#B8860B', label='NB model, one-step mean'); ax.fill_between(mu_nb.index, lo, hi, color='#B8860B', alpha=0.25, label='80% NB interval')
ax.axvline(pd.Timestamp('2026-01-01'), color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Weekly malaria cases: negative-binomial dynamic regression with Fourier seasonality and lagged rainfall')
cov = ((data['cases'][~train] >= lo) & (data['cases'][~train] <= hi)).mean()
print(f'80% interval coverage on 2026 (one-step): {cov:.2f}.  Rainfall (8-week lag) coefficient = {nb.params["rain_l8"]:.3f}, p = {nb.pvalues["rain_l8"]:.3f}')
_caption = 'A count model gives asymmetric, strictly non-negative intervals whose width grows with the level, and the 80% interval covers about 80% of the 2026 weeks. In this simulated series the lagged rainfall term adds nothing beyond the seasonal terms (the simulation\'s seasonality is calendar-driven); on a real surveillance series, whether rainfall carries information beyond seasonality is exactly the question Case Study 2 asks (the snapshot's provenance statement states whether the data are official or the course's stand-in).'

### 12.5 Anomaly and outbreak detection

In [ ]:
from statsmodels.tsa.seasonal import STL
y = cases[:'2025-12']
stl = STL(np.log1p(y), period=52, seasonal=13, robust=True).fit()
rem = stl.resid; mad = 1.4826 * np.median(np.abs(rem - np.median(rem)))
flag = rem.abs() > 3 * mad
# CUSUM on the standardised remainder for upward shifts
z = rem / mad; k, h_ = 0.5, 5; S = np.zeros(len(z)); alarm = np.zeros(len(z), bool)
for i in range(1, len(z)):
    S[i] = max(0, S[i-1] + z.iloc[i] - k); alarm[i] = S[i] > h_
    if alarm[i]: S[i] = 0
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
y.plot(ax=axes[0], lw=1, label='weekly cases'); axes[0].scatter(y.index[flag], y[flag], color='#A0302A', s=25, zorder=5, label='STL remainder > 3 robust s.d.'); axes[0].legend(fontsize=8); axes[0].set_title('Residual-threshold detection')
axes[1].plot(y.index, S, lw=1, label='CUSUM statistic'); axes[1].axhline(h_, color='#A0302A', lw=0.8, ls='--', label='alarm threshold h = 5'); axes[1].scatter(y.index[alarm], [h_] * alarm.sum(), color='#A0302A', s=25, zorder=5, label='alarms')
axes[1].legend(fontsize=8); axes[1].set_title('CUSUM detection of sustained upward shifts'); axes[1].set_xlabel('')
print('Weeks flagged by the 3-MAD threshold:', [d.date().isoformat() for d in y.index[flag]])
print('CUSUM alarms:', [d.date().isoformat() for d in y.index[alarm]])
_caption = 'Both detectors find the simulated 2022 outbreak. The point threshold fires on its largest weeks but also on isolated weeks in 2023 and 2024 (false alarms); the CUSUM fires only during the outbreak, once the excess has accumulated, and resets after each alarm. That contrast is the sensitivity-versus-false-alarm trade-off in one picture.'

## Exercises

1. Fit MSTL to the full 2014 half-hourly demand series with periods 48, 336 and an annual period approximated by Fourier terms; forecast one week ahead and compare with a weekly seasonal naive by rolling origin over December.
2. Build a day-ahead daily demand model with temperature (and its square), a work-day indicator and Fourier terms with ARIMA errors; evaluate it on rolling origins over the last quarter of 2014 against the model without temperature.
3. Implement the TSB variant of Croston (which updates the probability of demand rather than the interval) and compare with Croston on the simulated intermittent series.

In [ ]:
# Your work here
